In [1]:
"""
Classical ML Time Series Anomaly Detection Pipeline
- Point-wise anomaly detection (класс 1 = аномальная точка во времени)
- Группированный сплит по series_id для исключения data leakage
- Оптимизированные признаки и регуляризация для улучшения детекции редких аномалий
- Production-ready train/predict CLI
"""
import gc
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, f1_score, auc
import joblib
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import warnings
import argparse

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

DTYPE_FLOAT = np.float32

@dataclass
class Config:
    data_path: Path = Path("../data/R1.parquet")
    model_dir: Path = Path("models_point-wise")
    output_path: Optional[Path] = Path("results_point-wise")
    test_size: float = 0.15
    val_size: float = 0.15
    random_state: int = 42
    windows: Tuple[int, ...] = field(default=(10, 20, 50))
    lags: Tuple[int, ...] = field(default=(1, 3, 5))
    train_sample_frac: Optional[float] = 1.0

class FeatureEngineer:
    @staticmethod
    def compute_series_features(df: pd.DataFrame, config: Config) -> pd.DataFrame:
        res = pd.DataFrame(index=df.index)
        val = df['value'].astype(np.float32)

        # Мгновенные изменения
        res['diff_1'] = val.diff(1)
        res['diff_3'] = val.diff(3)
        res['abs_diff'] = res['diff_1'].abs()
        res['pct_change'] = val.pct_change(1)

        # Скользящие статистики + Z-score (локальные отклонения в единицах стандартного отклонения)
        for w in config.windows:
            roll = val.rolling(w, min_periods=1)
            res[f'roll_mean_{w}'] = roll.mean()
            res[f'roll_std_{w}'] = roll.std()
            res[f'z_score_{w}'] = (val - res[f'roll_mean_{w}']) / (res[f'roll_std_{w}'] + 1e-8)

        # Экспоненциально взвешенные признаки (быстрее реагируют на недавние аномалии)
        ewm = val.ewm(span=10, adjust=False)
        res['ewm_mean_10'] = ewm.mean()
        res['ewm_std_10'] = ewm.std()

        # Моменты распределения (асимметрия часто меняется перед/во время аномалии)
        res['roll_skew_10'] = val.rolling(10, min_periods=1).skew()
        res['roll_range_20'] = val.rolling(20, min_periods=1).max() - val.rolling(20, min_periods=1).min()

        # Авторегрессионные признаки
        for lag in config.lags:
            res[f'lag_{lag}'] = val.shift(lag)

        for col in res.select_dtypes(include=[np.number]).columns:
            res[col] = res[col].replace([np.inf, -np.inf], np.nan)

        res = res.astype({col: DTYPE_FLOAT for col in res.select_dtypes(include=[np.number]).columns})
        res['label'] = df['label'].astype(np.int8)
        res['series_id'] = df['series_id']
        res['time_index'] = df['time_index']
        return res

class AnomalyDetectorPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.feature_cols: List[str] = []
        self.pipeline: Optional[Pipeline] = None
        self.threshold: float = 0.5
        self._build_pipeline()

    def _build_pipeline(self):
        model = HistGradientBoostingClassifier(
            max_depth=6,
            learning_rate=0.05,
            max_iter=500,
            class_weight='balanced',
            random_state=self.config.random_state,
            validation_fraction=0.1,
            early_stopping=True,
            n_iter_no_change=20,
            max_bins=64,
            min_samples_leaf=50,
            l2_regularization=0.1
        )

        self.pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('model', model)
        ])

    def _extract_features(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info("Extracting features...")
        all_features = []
        batch_size = 20

        for batch_idx, (group_keys, group_data) in enumerate(self._batch_groups(df, 'series_id', batch_size)):
            batch_features = pd.concat([
                FeatureEngineer.compute_series_features(g, self.config)
                for _, g in group_data
            ], ignore_index=True)
            all_features.append(batch_features)
            del batch_features
            if batch_idx % 10 == 0:
                gc.collect()

        features = pd.concat(all_features, ignore_index=True)
        del all_features
        gc.collect()

        self.feature_cols = [c for c in features.columns if c not in ['label', 'series_id', 'time_index']]
        for col in self.feature_cols:
            if features[col].dtype == 'float64':
                features[col] = features[col].astype(DTYPE_FLOAT)

        nan_count = features[self.feature_cols].isna().sum().sum()
        if nan_count > 0:
            logger.warning(f"Found {nan_count} NaN values in features. Will be imputed with median.")

        logger.info(f"Features extracted: {features.shape}, Memory: {features.memory_usage(deep=True).sum() / 1e6:.2f} MB")
        return features

    @staticmethod
    def _batch_groups(df, col, batch_size):
        df_sorted = df.sort_values(col, ignore_index=True)
        unique_ids = df_sorted[col].unique()
        for i in range(0, len(unique_ids), batch_size):
            batch_keys = unique_ids[i:i+batch_size]
            mask = df_sorted[col].isin(batch_keys)
            batch_data = [(k, g) for k, g in df_sorted[mask].groupby(col, sort=False)]
            yield batch_keys, batch_data

    def _split_data(self, features: pd.DataFrame) -> Tuple:
        logger.info("Splitting data...")
        if features['series_id'].dtype != 'category':
            features['series_id'] = features['series_id'].astype('category')

        groups = features['series_id'].cat.codes.values
        unique_codes = np.unique(groups)

        gss = GroupShuffleSplit(n_splits=1, test_size=self.config.test_size, random_state=self.config.random_state)
        train_val_idx, test_idx = next(gss.split(unique_codes, groups=unique_codes))
        train_val_codes = unique_codes[train_val_idx]
        test_codes = unique_codes[test_idx]

        val_ratio = self.config.val_size / (1 - self.config.test_size)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=val_ratio, random_state=self.config.random_state)
        train_idx_final, val_idx_final = next(gss2.split(train_val_codes, groups=train_val_codes))

        train_codes = train_val_codes[train_idx_final]
        val_codes = train_val_codes[val_idx_final]

        mask_train = features['series_id'].cat.codes.isin(train_codes)
        mask_val = features['series_id'].cat.codes.isin(val_codes)
        mask_test = features['series_id'].cat.codes.isin(test_codes)

        X_train = features.loc[mask_train, self.feature_cols].values.astype(DTYPE_FLOAT)
        y_train = features.loc[mask_train, 'label'].values.astype(np.int8)
        X_val = features.loc[mask_val, self.feature_cols].values.astype(DTYPE_FLOAT)
        y_val = features.loc[mask_val, 'label'].values.astype(np.int8)
        X_test = features.loc[mask_test, self.feature_cols].values.astype(DTYPE_FLOAT)
        y_test = features.loc[mask_test, 'label'].values.astype(np.int8)

        del features, mask_train, mask_val, mask_test
        gc.collect()
        return X_train, y_train, X_val, y_val, X_test, y_test

    def _optimize_threshold(self, y_true: np.ndarray, y_prob: np.ndarray) -> float:
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        best_idx = np.argmax(f1_scores)
        best_thr = thresholds[min(best_idx, len(thresholds) - 1)]
        return float(np.clip(best_thr, 0.01, 0.99))

    def train(self, df: pd.DataFrame) -> Dict:
        if self.config.train_sample_frac and self.config.train_sample_frac < 1.0:
            logger.info(f"Sampling {self.config.train_sample_frac*100:.0f}% of data...")
            df = df.groupby('label', group_keys=False).sample(
                frac=self.config.train_sample_frac, random_state=self.config.random_state
            ).reset_index(drop=True)
            logger.info(f"After sampling: {len(df)} rows")
            if 'label' not in df.columns:
                raise ValueError("Label column lost after sampling.")

        features = self._extract_features(df)
        X_train, y_train, X_val, y_val, X_test, y_test = self._split_data(features)

        logger.info(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        logger.info(f"Train anomaly ratio: {y_train.mean():.4f}")

        self.pipeline.fit(X_train, y_train)

        val_prob = self.pipeline.predict_proba(X_val)[:, 1]
        self.threshold = self._optimize_threshold(y_val, val_prob)
        logger.info(f"Optimal threshold (F1): {self.threshold:.4f}")

        val_pred = (val_prob >= self.threshold).astype(int)
        logger.info("Validation metrics:\n" + classification_report(y_val, val_pred, zero_division=0))

        test_prob = self.pipeline.predict_proba(X_test)[:, 1]
        test_pred = (test_prob >= self.threshold).astype(int)

        precision_val = y_test[test_pred == 1].mean() if test_pred.sum() > 0 else 0.0
        recall_val = test_pred[y_test == 1].mean() if y_test.sum() > 0 else 0.0

        prec_curve, rec_curve, _ = precision_recall_curve(y_test, test_prob)
        pr_auc_val = auc(rec_curve, prec_curve)

        test_metrics = {
            'roc_auc': roc_auc_score(y_test, test_prob),
            'pr_auc': pr_auc_val,
            'f1': f1_score(y_test, test_pred, zero_division=0),
            'precision': precision_val,
            'recall': recall_val
        }
        logger.info("Test metrics:\n" + classification_report(y_test, test_pred, zero_division=0))
        logger.info(f"Test ROC-AUC: {test_metrics['roc_auc']:.4f}, PR-AUC: {test_metrics['pr_auc']:.4f}")
        return test_metrics

    def save(self):
        self.config.model_dir.mkdir(parents=True, exist_ok=True)
        artifacts = {
            'pipeline': self.pipeline,
            'feature_cols': self.feature_cols,
            'threshold': self.threshold,
            'config': self.config
        }
        path = self.config.model_dir / "anomaly_model.joblib"
        joblib.dump(artifacts, path)
        logger.info(f"Model saved to: {path}")

    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.pipeline is None:
            logger.warning("Model not initialized. Loading from disk...")
            self.load()

        features = self._extract_features(df)
        X = features[self.feature_cols]
        probs = self.pipeline.predict_proba(X)[:, 1]
        preds = (probs >= self.threshold).astype(int)

        out = df.copy()
        out['pred_label'] = preds
        out['pred_prob'] = probs
        return out

    def load(self):
        path = self.config.model_dir / "anomaly_model.joblib"
        if not path.exists():
            raise FileNotFoundError(f"Model not found: {path}")
        artifacts = joblib.load(path)
        self.pipeline = artifacts['pipeline']
        self.feature_cols = artifacts['feature_cols']
        self.threshold = artifacts['threshold']
        self.config = artifacts['config']
        logger.info("Model loaded successfully.")

def main():
    parser = argparse.ArgumentParser(description='Classical ML Anomaly Detection')
    parser.add_argument('mode', nargs='?', default='train', choices=['train', 'predict'], help='Execution mode')
    parser.add_argument('--data', type=Path, default=Path('../data/R1.parquet'), help='Path to data')
    parser.add_argument('--model-dir', type=Path, default=Path('models_point-wise'), help='Model directory')
    parser.add_argument('--output', type=Path, default=Path("results_point-wise"), help='Path for predictions')

    args, _ = parser.parse_known_args()
    config = Config(data_path=args.data, model_dir=args.model_dir, output_path=args.output)
    detector = AnomalyDetectorPipeline(config)

    if args.mode == 'train':
        logger.info("=== TRAINING ===")
        if not config.data_path.exists():
            raise FileNotFoundError(f"Data file not found: {config.data_path}")

        df = pd.read_parquet(config.data_path) if config.data_path.suffix == '.parquet' else pd.read_csv(config.data_path)

        # Convert PyArrow types to standard numpy types
        for col in df.columns:
            if pd.api.types.is_float_dtype(df[col]):
                df[col] = df[col].astype(np.float32)
            elif pd.api.types.is_integer_dtype(df[col]) and col != 'time_index':
                df[col] = df[col].astype(np.int32)
            if col == 'label':
                df[col] = df[col].astype(np.int8)
            if col == 'series_id':
                df[col] = df[col].astype('category')

        required = {'series_id', 'time_index', 'value', 'label'}
        if not required.issubset(df.columns):
            raise ValueError(f"Missing columns: {required - set(df.columns)}")

        metrics = detector.train(df)
        detector.save()
        logger.info(f"Final metrics: {metrics}")

    elif args.mode == 'predict':
        logger.info("=== PREDICTION ===")
        detector.load()
        df = pd.read_parquet(config.data_path) if config.data_path.suffix == '.parquet' else pd.read_csv(config.data_path)

        preds = detector.predict(df)
        out_path = config.output_path if config.output_path else Path("predictions.parquet")
        preds.to_parquet(out_path, index=False)
        logger.info(f"Predictions saved to: {out_path}")
        logger.info(f"Label distribution: {preds['pred_label'].value_counts().to_dict()}")

if __name__ == '__main__':
    main()

2026-05-23 15:44:47,350 | INFO     | === TRAINING ===
2026-05-23 15:44:48,722 | INFO     | Extracting features...
2026-05-23 15:46:32,726 | WARNING  | Found 507684 NaN values in features. Will be imputed with median.
2026-05-23 15:46:32,746 | INFO     | Features extracted: (12950281, 23), Memory: 1178.79 MB
2026-05-23 15:46:32,749 | INFO     | Splitting data...
2026-05-23 15:46:37,179 | INFO     | Train: 8970142, Val: 1973111, Test: 2007028
2026-05-23 15:46:37,187 | INFO     | Train anomaly ratio: 0.0516
2026-05-23 15:54:47,440 | INFO     | Optimal threshold (F1): 0.8635
2026-05-23 15:54:48,005 | INFO     | Validation metrics:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98   1873536
           1       0.63      0.58      0.60     99575

    accuracy                           0.96   1973111
   macro avg       0.80      0.78      0.79   1973111
weighted avg       0.96      0.96      0.96   1973111

2026-05-23 15:55:21,325 | INFO     | T